In [4]:
import os
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

DIR_PACKAGE = Path.cwd().parent
DIR_REPO = DIR_PACKAGE.parent
if str(DIR_REPO) not in sys.path:
    sys.path.insert(0, str(DIR_REPO))

from sim_data_analyzer.xr_adapters import get_lfp_xr, get_net_rate_dynamics_xr
from sim_data_analyzer.xr_signal import interp_time_outliers
from sim_data_analyzer.xr_spect import calc_xr_cpsd, calc_xr_tf, calc_xr_welch
from sim_data_analyzer.xr_io import save_xr, load_xr

In [5]:
EXP_SRC = ('a1_lfp_15s', 'data_00000_seed_1000.pkl')
EXP_LABEL = 'a1_lfp_15s_0'

RATE_DT = 0.005

In [ ]:
dirpath_src = DIR_PACKAGE / 'dev_scratch' / 'data_src'
dirpath_proc = DIR_PACKAGE / 'dev_scratch' / 'data_proc'
os.makedirs(dirpath_proc, exist_ok=True)

lfp_loaded = False
rates_loaded = False

# Try to load pre-extracted lfp
fpath_lfp = dirpath_proc / f'{EXP_LABEL}_lfp.nc'
if fpath_lfp.exists():
    lfp = load_xr(fpath_lfp)
    lfp_loaded = True
    print(f'Loaded LFP from {fpath_lfp}')

# Try to load pre-extracted rates
fpath_rates = dirpath_proc / f'{EXP_LABEL}_rates_dt_{RATE_DT}.nc'
if fpath_rates.exists():
    rates = load_xr(fpath_rates)
    rates_loaded = True
    print (f'Loaded rates from {fpath_rates}')

# If needed - load simulation result and extract lfp/rates
if (not lfp_loaded) or (not rates_loaded):
    # Load simulation result from pkl
    print (f'Loading sim result from {fpath_sim_res}...')
    fpath_sim_res = dirpath_src / EXP_SRC[0] / EXP_SRC[1]
    with fpath_sim_res.open('rb') as fid:
        sim_result = pickle.load(fid)
    print (f'Loaded sim result from {fpath_sim_res}')

    # Extract LFP
    if not lfp_loaded:
        lfp = get_lfp_xr(sim_result)
        save_xr(lfp, fpath_lfp)
        print(f'Saved LFP to {fpath_lfp}')
    
    # Extract rates
    if not rates_loaded:
        rates = get_net_rate_dynamics_xr(sim_result, dt=RATE_DT)
        save_xr(rates, fpath_rates)
        print(f'Saved rates to {fpath_rates}')
